In [ ]:
import numpy as np
import torch
import time
import os
import csv

from datasets import load_dataset, concatenate_datasets
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
)


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val= load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test= load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")
print(train)  # Print the first example to understand its structure

train_df = train.to_pandas()
print("Sample data (first 5 rows):")
print(train_df.head())

full_data = concatenate_datasets([train, val, test])

# Total size
len_total = len(full_data)

# Exact 70/20/10 counts
target_train= int(round(0.70 * len_total))
target_val   = int(round(0.20 * len_total))
target_test  = len_total - target_train - target_val

split_1 = full_data.train_test_split(train_size=target_train, seed=42)
train_data = split_1["train"]
remaining  = split_1["test"]

# Split remaining into val and test
split_2 = remaining.train_test_split(train_size=target_val, seed=42)
val_data  = split_2["train"]
test_data = split_2["test"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))



Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

eng/train-00000-of-00001.parquet:   0%|          | 0.00/179k [00:00<?, ?B/s]

eng/dev-00000-of-00001.parquet:   0%|          | 0.00/11.8k [00:00<?, ?B/s]

eng/test-00000-of-00001.parquet:   0%|          | 0.00/182k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2763 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/115 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2765 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise'],
    num_rows: 2763
})
Sample data (first 5 rows):
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1         2  
Train size: 3950
Validation size: 1129
Test size: 564


In [3]:
TEXT_COL = "text"

EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]

In [4]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=256,
    )
    encoded["labels"] = [float(example[e]) for e in EMOTIONS]
    return encoded

train_tok = train_data.map(preprocess)
val_tok   = val_data.map(preprocess)
test_tok  = test_data.map(preprocess)

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1129 [00:00<?, ? examples/s]

Map:   0%|          | 0/564 [00:00<?, ? examples/s]

In [ ]:
cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

In [6]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(EMOTIONS),
    problem_type="regression"
).to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def safe_pearson(x, y):
    r, _ = pearsonr(x, y)
    return 0.0 if np.isnan(r) else float(r)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds)
    labels = np.asarray(labels)

    metrics = {}
    rs = []

    for i, emo in enumerate(EMOTIONS):
        r = safe_pearson(preds[:, i], labels[:, i])
        metrics[f"pearson_{emo}"] = r
        rs.append(r)

    metrics["pearson_mean"] = float(np.mean(rs))
    return metrics

In [ ]:
LOG_FILE = "roberta_loss_accuracy.csv"

if os.path.exists(LOG_FILE):
    print(f"CSV file exists at: {LOG_FILE}")
else:
    print("CSV file not found. It will be created during training.")

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "eval_loss", "accuracy"])  # accuracy = pearson_mean

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.last_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            epoch = metrics.get("epoch", state.epoch)
            eval_loss = metrics.get("eval_loss", "")
            accuracy = metrics.get("eval_pearson_mean", "")

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([epoch, self.last_train_loss, eval_loss, accuracy])


In [14]:
EPOCHS = 100

training_args = TrainingArguments(
    output_dir="roberta_brighter_onlyintensities",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=EPOCHS,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="pearson_mean",
    greater_is_better=True,

    report_to="none",
    fp16=torch.cuda.is_available(),
)

# ==============================
# 12) Trainer
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(LOG_FILE)],
)

# ==============================
# 13) Train
# ==============================
start = time.time()
trainer.train()
end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / EPOCHS:.1f} seconds")




Epoch,Training Loss,Validation Loss,Pearson Anger,Pearson Fear,Pearson Joy,Pearson Sadness,Pearson Surprise,Pearson Mean
1,0.106046,0.695098,0.137054,0.244795,0.223913,0.223453,0.272558,0.220355


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)

In [10]:
print("CSV log saved as:", LOG_FILE)

CSV log saved as: roberta_loss_accuracy.csv


In [11]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {"raw": float(logits[i]), "intensity_0_3": int(discrete[i])}
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))

{'anger': {'raw': 0.345703125, 'intensity_0_3': 0}, 'fear': {'raw': 0.625, 'intensity_0_3': 1}, 'joy': {'raw': 0.8740234375, 'intensity_0_3': 1}, 'sadness': {'raw': 0.25048828125, 'intensity_0_3': 0}, 'surprise': {'raw': 0.88818359375, 'intensity_0_3': 1}}
